---
title: "Predict restaurant ratings with TabPFN 3.5"
description: "Compare TabPFN 3.5, TabPFN 3, and five-minute-tuned XGBoost on Zomato text and tabular data."
icon: utensils
cookbookTags:
  - benchmark
  - regression
  - text
authors:
  - name: Prior Labs
---
Predict Zomato restaurant ratings from structured fields and text. All three models receive the same source columns, 10,000 training rows, and 2,000 held-out rows. XGBoost gets a **300-second hyperparameter search**; neither TabPFN model is tuned.

TabPFN handles raw text through the staging API. XGBoost treats every string column, including text, as a native categorical feature. This uses the Zomato dataset from [MulTaBench](https://github.com/alanarazi7/MulTaBench), with one smaller split and different preprocessing; scores are not the five-fold leaderboard results.

## Setup
Use a CPU runtime and a staging API token with TabPFN 3.5 enabled. Set `TABPFN_CLIENT_API_URL` **before importing the client**. The client revision below supports `ModelVersion.V3_5`; staging models may change before release.

In [ ]:
%pip install -q "tabpfn-client @ git+https://github.com/PriorLabs/tabpfn-client.git@1227e3f07ec8538637c4fbb728c8eba1931dd9fc" "xgboost==2.1.4" "optuna==4.9.0" "scikit-learn==1.6.1" "pandas==2.3.3"

In [1]:
import gc
import os
import time
from getpass import getpass
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from IPython.display import display
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split

os.environ["TABPFN_CLIENT_API_URL"] = "https://api.stg.priorlabs.ai"
if not os.environ.get("TABPFN_TOKEN"):
    os.environ["TABPFN_TOKEN"] = getpass("Staging API token: ")
from tabpfn_client import TabPFNRegressor
from tabpfn_client.api_models import ModelVersion

SEED = 42
DEVICE = "cpu"
print("TabPFN: raw text via staging; XGBoost: native categories on CPU")

TabPFN: raw text via staging; XGBoost: native categories on CPU


## Load and split
The [curated MulTaBench dataset](https://www.kaggle.com/datasets/chico89/multabench-zomato-restaurants) contains 41,665 rated listings from the [original Zomato dataset](https://www.kaggle.com/datasets/himanshupoddar/zomato-bangalore-restaurants). Ratings are numeric and URLs have already been removed. Reserve the test set first, then sample 10,000 training rows and hold out 1,000 of those for tuning.

This is a random **listing-level** split: restaurants can recur across splits, and `reviews_list` contains individual review scores. Interpret this as predicting existing listings' aggregate ratings, not performance on unseen restaurants or future ratings.

In [2]:
data_dir = Path("data/zomato")
data_dir.mkdir(parents=True, exist_ok=True)
if not (data_dir / "data.csv").exists():
    urlretrieve(
        "https://www.kaggle.com/api/v1/datasets/download/chico89/"
        "multabench-zomato-restaurants?datasetVersionNumber=1",
        data_dir / "dataset.zip",
    )
    with ZipFile(data_dir / "dataset.zip") as archive:
        (data_dir / "data.csv").write_bytes(archive.read("data.csv"))
df = pd.read_csv(data_dir / "data.csv")
assert len(df) == 41_665 and df["rate"].between(0, 5).all()
pool, test = train_test_split(df, test_size=2_000, random_state=SEED)
train = pool.sample(n=10_000, random_state=SEED)
fit_idx, val_idx = train_test_split(np.arange(len(train)), test_size=1_000, random_state=SEED)
y_train, y_test = train["rate"].to_numpy(), test["rate"].to_numpy()
features = pd.concat([train, test], ignore_index=True).drop(columns="rate")
features["approx cost(for two people)"] = pd.to_numeric(
    features["approx cost(for two people)"].astype(str).str.replace(",", "", regex=False), errors="coerce"
)
print(f"Train: {len(train):,} (including {len(val_idx):,} validation); test: {len(test):,}")

Train: 10,000 (including 1,000 validation); test: 2,000


In [3]:
X_train, X_test = features.iloc[:len(train)], features.iloc[len(train):]
results = []
predictions = {}

def record(name, prediction):
    predictions[name] = prediction
    results.append({
        "Model": name,
        "R² ↑": r2_score(y_test, prediction),
        "RMSE ↓": root_mean_squared_error(y_test, prediction),
        "MAE ↓": mean_absolute_error(y_test, prediction),
    })



## XGBoost: native categorical features
Treat every string column, including reviews, as a pandas category and set `enable_categorical=True`. Categories come from the inner training set; unseen validation/test values become missing. Numeric missing values are handled natively.

In [4]:
preprocess_start = time.perf_counter()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()
X_train_xgb, X_test_xgb = X_train.copy(), X_test.copy()
for col in cat_cols:
    dtype = pd.CategoricalDtype(categories=X_train.iloc[fit_idx][col].dropna().unique())
    X_train_xgb[col] = X_train[col].astype(dtype)
    X_test_xgb[col] = X_test[col].astype(dtype)
preprocess_seconds = time.perf_counter() - preprocess_start
print(f"XGBoost: {len(cat_cols)} categorical columns; preprocessing: {preprocess_seconds:.1f}s")

XGBoost: 13 categorical columns; preprocessing: 0.4s


Optuna selects hyperparameters by validation RMSE; early stopping selects the tree count. A callback stops the active trial at the 300-second deadline (checked after each boosting iteration). Refit on all 10,000 training rows. Refitting and prediction are outside the tuning budget.

In [5]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
xgb_common = dict(objective="reg:squarederror", eval_metric="rmse",
                  tree_method="hist", device=DEVICE, enable_categorical=True,
                  n_jobs=8, random_state=SEED)
start = time.perf_counter()
deadline = start + 300

class Deadline(xgb.callback.TrainingCallback):
    def after_iteration(self, model, epoch, evals_log):
        return time.perf_counter() >= deadline

def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 100, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 100, log=True),
    }
    model = xgb.XGBRegressor(**xgb_common, **params, n_estimators=3_000,
                            early_stopping_rounds=50, callbacks=[Deadline()])
    model.fit(X_train_xgb.iloc[fit_idx], y_train[fit_idx],
              eval_set=[(X_train_xgb.iloc[val_idx], y_train[val_idx])], verbose=False)
    trial.set_user_attr("n_estimators", model.best_iteration + 1)
    return float(model.best_score)

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, timeout=max(0, deadline - time.perf_counter()))
tuning_seconds = time.perf_counter() - start
best_params = dict(study.best_params, n_estimators=study.best_trial.user_attrs["n_estimators"])
model = xgb.XGBRegressor(**xgb_common, **best_params)
model.fit(X_train_xgb, y_train, verbose=False)
prediction = model.predict(X_test_xgb)
record("XGBoost (5-min tuning)", prediction)
print(f"Search: {tuning_seconds:.1f}s; trials: {len(study.trials)}; trees: {best_params['n_estimators']}")
print(best_params)

Search: 301.4s; trials: 27; trees: 357
{'max_depth': 9, 'learning_rate': 0.05412239278765842, 'min_child_weight': 1.0231860371088157, 'subsample': 0.9502760624272271, 'colsample_bytree': 0.8081445839369573, 'reg_alpha': 3.365484610918805e-08, 'reg_lambda': 0.5497054362823262, 'n_estimators': 357}


## TabPFN 3.5 and TabPFN 3
Pass the original columns, including review text, directly to the client. Staging handles text processing. Use `ModelVersion` to select each version, with eight ensemble members and no tuning.

In [6]:
common = dict(n_estimators=8, random_state=SEED)
for name, version in [("TabPFN 3.5", ModelVersion.V3_5), ("TabPFN 3", ModelVersion.V3)]:
    model = TabPFNRegressor.create_default_for_version(version, **common)
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    record(name, prediction)
    print(f"{name}: R²={results[-1]['R² ↑']:.4f}")
    del model
    _ = gc.collect()

TabPFN 3.5: R²=0.8957
TabPFN 3: R²=0.8836


## Compare
R² is higher-is-better; RMSE and MAE are in rating points. This is one fixed split, not a significance claim. TabPFN handles text remotely; XGBoost treats text as category labels on CPU.

In [7]:
display(pd.DataFrame(results).set_index("Model").round(4))

,R² ↑,RMSE ↓,MAE ↓
Model,,,
XGBoost (5-min tuning),0.8355,0.1784,0.0998
TabPFN 3.5,0.8957,0.1420,0.0614
TabPFN 3,0.8836,0.1500,0.0701
